# Austin Ionosonde Data Processing - Figure 8

This notebook loads the trusted manually-scaled ionosonde data from Austin, TX (AU930),
converts it to a clean CSV format, and creates visualization plots.

In [ ]:
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.dates as mdates

%matplotlib inline

## Load GIRO DIDBase TXT File

The data file is in GIRO DIDBase format with manually scaled measurements by David Themens.
It includes:
- foF2, foF1, foE: Critical frequencies of the F2, F1, and E layers (MHz)
- hmF2, hmF1, hmE: Virtual heights of the F2, F1, and E layers (km)
- Missing values are denoted as '---' in the file

In [ ]:
# Path to the trusted TXT data file
txt_path = '../data/CSVs/2024-04-08_AU930_AustinTX_Ionosonde_ManualScaled.TXT'

# Read file and extract header lines and data
header_lines = []
data_lines = []
with open(txt_path, 'r') as f:
    for line in f:
        line_stripped = line.strip()
        # Save header/comment lines
        if line_stripped.startswith('#'):
            header_lines.append(line_stripped)
            continue
        # Skip column header line and empty lines
        if 'yyyy.MM.dd' in line_stripped or not line_stripped:
            continue
        # Skip summary statistics at the end
        if any(keyword in line_stripped for keyword in ['Ql', 'Med', 'Qu', 'Avg', 'Min', 'Max']):
            continue
        data_lines.append(line_stripped)

print(f"Found {len(header_lines)} header lines")
print(f"Found {len(data_lines)} data records")

# Parse the data lines
records = []
for line in data_lines:
    parts = line.split()
    if len(parts) < 9:
        print(f"Skipping line with {len(parts)} parts: {line}")
        continue
    
    # Parse datetime: yyyy.MM.dd (DDD) HH:mm:ss
    # parts[0] = yyyy.MM.dd, parts[1] = (DDD), parts[2] = HH:mm:ss
    date_str = f"{parts[0]} {parts[2]}"
    try:
        dt = pd.to_datetime(date_str, format='%Y.%m.%d %H:%M:%S')
    except Exception as e:
        print(f"Error parsing datetime '{date_str}': {e}")
        continue
    
    # Helper function to parse values, replacing '---' with NaN
    def parse_val(s):
        try:
            return float(s) if s != '---' else np.nan
        except:
            return np.nan
    
    # Column indices after date/time:
    # parts[3] = C-score
    # parts[4] = foF2
    # parts[5] = foF1
    # parts[6] = foE
    # parts[7] = hmF2
    # parts[8] = hmF1
    # parts[9] = hmE
    record = {
        'UTC': dt,
        'foF2': parse_val(parts[4]) if len(parts) > 4 else np.nan,
        'foF1': parse_val(parts[5]) if len(parts) > 5 else np.nan,
        'foE': parse_val(parts[6]) if len(parts) > 6 else np.nan,
        'hmF2': parse_val(parts[7]) if len(parts) > 7 else np.nan,
        'hmF1': parse_val(parts[8]) if len(parts) > 8 else np.nan,
        'hmE': parse_val(parts[9]) if len(parts) > 9 else np.nan,
    }
    records.append(record)

# Create DataFrame
ionosonde_df = pd.DataFrame(records)

print(f"\nDataFrame shape: {ionosonde_df.shape}")
print(f"Columns: {ionosonde_df.columns.tolist()}")
print(f"\nDate range: {ionosonde_df['UTC'].min()} to {ionosonde_df['UTC'].max()}")
print(f"\nFirst few rows:")
display(ionosonde_df.head(10))
print(f"\nData summary:")
display(ionosonde_df.describe())

## Save to CSV Format

Save the cleaned data to a CSV file for easier use in other notebooks.

In [ ]:
# Save to CSV with header information preserved
csv_output_path = '../data/CSVs/2024-04-08_AU930_AustinTX_Ionosonde_ManualScaled.csv'

# Write header lines as comments, then append the DataFrame
with open(csv_output_path, 'w') as f:
    # Write all header lines from the TXT file as comments
    for header_line in header_lines:
        f.write(f"{header_line}\n")
    f.write("#\n")
    f.write("# Converted to CSV format for easier processing\n")
    f.write("#\n")

# Append the DataFrame to the file
ionosonde_df.to_csv(csv_output_path, mode='a', index=False)

print(f"Saved cleaned data to: {csv_output_path}")
print(f"File size: {os.path.getsize(csv_output_path) / 1024:.2f} KB")
print(f"Header lines preserved: {len(header_lines)}")

# Test reloading the CSV (comment='#' skips the header lines)
test_df = pd.read_csv(csv_output_path, comment='#', parse_dates=['UTC'])
print(f"\nTest reload successful: {test_df.shape}")
print(f"Columns: {test_df.columns.tolist()}")
assert ionosonde_df.shape == test_df.shape, "Shape mismatch!"
print("✓ CSV file verified!")
print("✓ Header information preserved as comments")

## Plot Virtual Heights

Plot hmF2, hmF1, and hmE (virtual heights of ionospheric layers).

In [ ]:
# Setup plotting style
import matplotlib as mpl
mpl.rcParams['font.size'] = 16
mpl.rcParams['font.weight'] = 'bold'
mpl.rcParams['axes.labelweight'] = 'bold'
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['axes.grid'] = True
mpl.rcParams['grid.linestyle'] = ':'

fig, ax = plt.subplots(figsize=(16, 9))

# Plot virtual heights
ax.plot(ionosonde_df['UTC'], ionosonde_df['hmF2'], 
        label='hmF2', marker='o', linestyle='-', linewidth=2, color='purple')
ax.plot(ionosonde_df['UTC'], ionosonde_df['hmF1'], 
        label='hmF1', marker='s', linestyle='--', linewidth=2, color='blue')
ax.plot(ionosonde_df['UTC'], ionosonde_df['hmE'], 
        label='hmE', marker='^', linestyle='-.', linewidth=2, color='brown')

# Format the plot
ax.set_xlabel('Time UTC')
ax.set_ylabel('Virtual Height [km]')
ax.set_title('Austin Ionosonde - Virtual Heights\n2024-04-08', fontsize=26)
ax.legend(loc='best', fontsize=14)
ax.set_ylim(50, 450)

# Format datetime axis
myFmt = mdates.DateFormatter('%H:%M')
ax.xaxis.set_major_formatter(myFmt)
ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=60))
fig.autofmt_xdate()

plt.tight_layout()
plt.savefig('fig_08a_virtual_heights.jpg', dpi=300, bbox_inches='tight', format='jpeg', pil_kwargs={'quality': 95})
print("Saved: fig_08a_virtual_heights.jpg")
plt.show()

## Plot Critical Frequencies

Plot foF2, foF1, and foE (critical frequencies of ionospheric layers).

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))

# Plot critical frequencies
ax.plot(ionosonde_df['UTC'], ionosonde_df['foF2'], 
        label='foF2', marker='o', linestyle='-', linewidth=2, color='darkgreen')
ax.plot(ionosonde_df['UTC'], ionosonde_df['foF1'], 
        label='foF1', marker='s', linestyle='--', linewidth=2, color='orange')
ax.plot(ionosonde_df['UTC'], ionosonde_df['foE'], 
        label='foE', marker='^', linestyle='-.', linewidth=2, color='red')

# Format the plot
ax.set_xlabel('Time UTC')
ax.set_ylabel('Critical Frequency [MHz]')
ax.set_title('Austin Ionosonde - Critical Frequencies\n2024-04-08', fontsize=26)
ax.legend(loc='best', fontsize=14)
ax.set_ylim(0, 12)

# Format datetime axis
myFmt = mdates.DateFormatter('%H:%M')
ax.xaxis.set_major_formatter(myFmt)
ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=60))
fig.autofmt_xdate()

plt.tight_layout()
plt.savefig('fig_08b_critical_frequencies.jpg', dpi=300, bbox_inches='tight', format='jpeg', pil_kwargs={'quality': 95})
print("Saved: fig_08b_critical_frequencies.jpg")
plt.show()